# BHAVYA CHOPRA
# Retail Analytics & AI-Powered Sales Forecasting System

**Major Project — End-to-End Data Science Solution**

This project analyzes multi-store retail sales, identifies business patterns, forecasts future revenue using time-series modelling, segments stores using machine learning, and generates management-ready insights.

### Main deliverables
1. Data loading and quality checks  
2. Exploratory data analysis  
3. KPI and trend analysis  
4. Time-series revenue forecasting  
5. Store performance segmentation using K-Means  
6. AI-style automatic business recommendations  
7. Export of final results for dashboard use

## 1. Install and import libraries


In [ ]:

import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, mean_absolute_error, mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Libraries imported successfully.')

Libraries imported successfully.


2. Insert Dataset

In [ ]:
#upload data set
try:
    from google.colab import files
    uploaded = files.upload()
    file_name = next(iter(uploaded))
except Exception:
    file_name = 'data/Retail_Sales_Data.csv'

if file_name.lower().endswith('.xlsx'):
    df = pd.read_excel(file_name)
else:
    df = pd.read_csv(file_name)

print('Dataset loaded successfully!')
print('Shape:', df.shape)
display(df.head())

Saving 4877028-Retail_Sales_Data.xlsx to 4877028-Retail_Sales_Data (1).xlsx
Dataset loaded successfully!
Shape: (73000, 19)


,Date,Store_ID,Store_Location,Product_ID,Product_Category,Product_Subcategory,Brand,Unit_Price,Units_Sold,Total_Sales,Discount_Percentage,Revenue,Customer_Type,Payment_Mode,Promotion_Applied,Stock_On_Hand,Store_Rating,Region,Holiday_Flag
0,13-04-2023,STR_104,Chennai,PRD_072,Sports,Athletics,Reebok,"29,973.06",39,"1,168,949.34",5,"1,110,501.87",Returning,Debit Card,Yes,422,4.40,South,0
1,25-10-2024,STR_103,Delhi,PRD_492,Sports,Outdoor,Yonex,"46,933.78",2,"93,867.56",15,"79,787.43",Returning,UPI,No,302,4.10,North,0
2,28-02-2023,STR_107,Kolkata,PRD_958,Groceries,Household,Nestle,"39,280.28",44,"1,728,332.32",15,"1,469,082.47",Returning,Cash,No,184,3.80,East,0
3,16-06-2023,STR_102,Bangalore,PRD_014,Home Appliances,Kitchen,Whirlpool,"40,439.03",9,"363,951.27",5,"345,753.71",New,UPI,Yes,493,3.70,South,0
4,23-05-2024,STR_108,Ahmedabad,PRD_932,Fashion,Women Clothing,Puma,"9,193.58",4,"36,774.32",5,"34,935.60",Returning,Credit Card,No,359,3.50,West,0


## 3. Data understanding and cleaning

In [ ]:
print('Column names:')
print(df.columns.tolist())
print('Data types:')
display(df.dtypes.to_frame('Data Type'))

print('Missing values:')
display(df.isnull().sum().to_frame('Missing Values'))

print('\nDuplicate rows:', df.duplicated().sum())

Column names:
['Date', 'Store_ID', 'Store_Location', 'Product_ID', 'Product_Category', 'Product_Subcategory', 'Brand', 'Unit_Price', 'Units_Sold', 'Total_Sales', 'Discount_Percentage', 'Revenue', 'Customer_Type', 'Payment_Mode', 'Promotion_Applied', 'Stock_On_Hand', 'Store_Rating', 'Region', 'Holiday_Flag']
Data types:


,Data Type
Date,object
Store_ID,object
Store_Location,object
Product_ID,object
Product_Category,object
Product_Subcategory,object
Brand,object
Unit_Price,float64
Units_Sold,int64
Total_Sales,float64


Missing values:


,Missing Values
Date,0
Store_ID,0
Store_Location,0
Product_ID,0
Product_Category,0
Product_Subcategory,0
Brand,0
Unit_Price,0
Units_Sold,0
Total_Sales,0



Duplicate rows: 0


In [ ]:
# Convert date and standardize important fields
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')

# Remove invalid/duplicate rows only if they exist
df = df.dropna(subset=['Date', 'Revenue', 'Units_Sold']).drop_duplicates().copy()

# Ensure numeric columns are valid
numeric_columns = [
    'Unit_Price', 'Units_Sold', 'Total_Sales', 'Discount_Percentage',
    'Revenue', 'Stock_On_Hand', 'Store_Rating', 'Holiday_Flag'
]
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=numeric_columns)

# Feature engineering
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.month_name().str[:3]
df['Quarter'] = 'Q' + df['Date'].dt.quarter.astype(str)
df['Day_Name'] = df['Date'].dt.day_name()
df['Year_Month'] = df['Date'].dt.to_period('M').astype(str)
df['Profit_After_Discount'] = df['Total_Sales'] - df['Revenue']
df['Revenue_Per_Unit'] = np.where(df['Units_Sold'] > 0, df['Revenue'] / df['Units_Sold'], 0)

print('Cleaned shape:', df.shape)
print('Date range:', df['Date'].min().date(), 'to', df['Date'].max().date())
print('Number of stores:', df['Store_ID'].nunique())
print('Number of product categories:', df['Product_Category'].nunique())
display(df.head())

Cleaned shape: (73000, 27)
Date range: 2023-01-01 to 2024-12-31
Number of stores: 8
Number of product categories: 5


,Date,Store_ID,Store_Location,Product_ID,Product_Category,Product_Subcategory,Brand,Unit_Price,Units_Sold,Total_Sales,Discount_Percentage,Revenue,Customer_Type,Payment_Mode,Promotion_Applied,Stock_On_Hand,Store_Rating,Region,Holiday_Flag,Year,Month,Month_Name,Quarter,Day_Name,Year_Month,Profit_After_Discount,Revenue_Per_Unit
0,2023-04-13,STR_104,Chennai,PRD_072,Sports,Athletics,Reebok,"29,973.06",39,"1,168,949.34",5,"1,110,501.87",Returning,Debit Card,Yes,422,4.40,South,0,2023,4,Apr,Q2,Thursday,2023-04,"58,447.47","28,474.41"
1,2024-10-25,STR_103,Delhi,PRD_492,Sports,Outdoor,Yonex,"46,933.78",2,"93,867.56",15,"79,787.43",Returning,UPI,No,302,4.10,North,0,2024,10,Oct,Q4,Friday,2024-10,"14,080.13","39,893.71"
2,2023-02-28,STR_107,Kolkata,PRD_958,Groceries,Household,Nestle,"39,280.28",44,"1,728,332.32",15,"1,469,082.47",Returning,Cash,No,184,3.80,East,0,2023,2,Feb,Q1,Tuesday,2023-02,"259,249.85","33,388.24"
3,2023-06-16,STR_102,Bangalore,PRD_014,Home Appliances,Kitchen,Whirlpool,"40,439.03",9,"363,951.27",5,"345,753.71",New,UPI,Yes,493,3.70,South,0,2023,6,Jun,Q2,Friday,2023-06,"18,197.56","38,417.08"
4,2024-05-23,STR_108,Ahmedabad,PRD_932,Fashion,Women Clothing,Puma,"9,193.58",4,"36,774.32",5,"34,935.60",Returning,Credit Card,No,359,3.50,West,0,2024,5,May,Q2,Thursday,2024-05,"1,838.72","8,733.90"


## 4. Summary statistics and management KPIs

In [ ]:
total_revenue = df['Revenue'].sum()
total_sales = df['Total_Sales'].sum()
total_units = df['Units_Sold'].sum()
avg_order_value = df['Revenue'].mean()
avg_discount = df['Discount_Percentage'].mean()

kpi_table = pd.DataFrame({
    'KPI': ['Total Revenue', 'Gross Sales', 'Total Units Sold', 'Average Transaction Revenue', 'Average Discount %'],
    'Value': [total_revenue, total_sales, total_units, avg_order_value, avg_discount]
})
display(kpi_table)

print(f'Total Revenue: ₹{total_revenue:,.2f}')
print(f'Gross Sales: ₹{total_sales:,.2f}')
print(f'Total Units Sold: {total_units:,.0f}')
print(f'Average Revenue per Record: ₹{avg_order_value:,.2f}')
print(f'Average Discount: {avg_discount:.2f}%')

,KPI,Value
0,Total Revenue,"41,133,782,865.04"
1,Gross Sales,"45,708,091,467.57"
2,Total Units Sold,"1,817,305.00"
3,Average Transaction Revenue,"563,476.48"
4,Average Discount %,10.01


Total Revenue: ₹41,133,782,865.04
Gross Sales: ₹45,708,091,467.57
Total Units Sold: 1,817,305
Average Revenue per Record: ₹563,476.48
Average Discount: 10.01%


## 5. Exploratory Data Analysis

In [ ]:
# Monthly revenue trend
monthly_revenue = (
    df.groupby(pd.Grouper(key='Date', freq='MS'))['Revenue']
      .sum().reset_index()
)
fig = px.line(monthly_revenue, x='Date', y='Revenue', markers=True,
              title='Monthly Revenue Trend')
fig.update_layout(yaxis_title='Revenue (₹)', xaxis_title='Month')
fig.show()

In [ ]:
# Revenue by product category
category_summary = (df.groupby('Product_Category')
                      .agg(Revenue=('Revenue','sum'), Units_Sold=('Units_Sold','sum'))
                      .sort_values('Revenue', ascending=False).reset_index())
display(category_summary)

fig = px.bar(category_summary, x='Product_Category', y='Revenue', text_auto='.3s',
             title='Revenue by Product Category')
fig.update_layout(yaxis_title='Revenue (₹)', xaxis_title='Category')
fig.show()

,Product_Category,Revenue,Units_Sold
0,Electronics,"8,318,538,868.88",367921
1,Home Appliances,"8,262,751,184.02",365113
2,Fashion,"8,215,645,302.68",361431
3,Groceries,"8,170,528,429.92",359711
4,Sports,"8,166,319,079.53",363129


In [ ]:
# Store performance
store_summary = (df.groupby(['Store_ID','Store_Location','Region'])
                   .agg(Revenue=('Revenue','sum'), Units_Sold=('Units_Sold','sum'),
                        Avg_Rating=('Store_Rating','mean'), Avg_Discount=('Discount_Percentage','mean'),
                        Transactions=('Revenue','size'))
                   .sort_values('Revenue', ascending=False).reset_index())
display(store_summary)

fig = px.bar(store_summary, x='Store_Location', y='Revenue', color='Region', text_auto='.3s',
             title='Store-wise Revenue Performance')
fig.show()

,Store_ID,Store_Location,Region,Revenue,Units_Sold,Avg_Rating,Avg_Discount,Transactions
0,STR_103,Delhi,North,"5,223,064,263.84",229475,4.25,9.97,9173
1,STR_102,Bangalore,South,"5,186,276,605.87",228384,4.25,10.00,9174
2,STR_101,Mumbai,West,"5,169,675,266.91",226670,4.25,10.09,9039
3,STR_104,Chennai,South,"5,165,937,477.75",228580,4.25,10.11,9186
4,STR_105,Pune,West,"5,161,898,681.92",227715,4.25,10.08,9254
5,STR_107,Kolkata,East,"5,107,832,301.73",229403,4.25,9.95,9136
6,STR_108,Ahmedabad,West,"5,078,321,969.54",223994,4.25,9.94,9024
7,STR_106,Hyderabad,South,"5,040,776,297.48",223084,4.25,9.94,9014


In [ ]:
# Region, customer, payment and promotion analysis
for column in ['Region', 'Customer_Type', 'Payment_Mode', 'Promotion_Applied']:
    summary = df.groupby(column)['Revenue'].sum().sort_values(ascending=False).reset_index()
    fig = px.bar(summary, x=column, y='Revenue', text_auto='.3s',
                 title=f'Revenue by {column.replace("_", " ")}')
    fig.show()

In [ ]:
# Holiday versus non-holiday performance
holiday_analysis = df.groupby('Holiday_Flag').agg(
    Total_Revenue=('Revenue','sum'),
    Average_Revenue=('Revenue','mean'),
    Units_Sold=('Units_Sold','sum'),
    Records=('Revenue','size')
).reset_index()
holiday_analysis['Day_Type'] = holiday_analysis['Holiday_Flag'].map({0:'Non-Holiday',1:'Holiday'})
display(holiday_analysis)

,Holiday_Flag,Total_Revenue,Average_Revenue,Units_Sold,Records,Day_Type
0,0,"40,640,914,750.14","563,650.82",1795915,72103,Non-Holiday
1,1,"492,868,114.90","549,462.78",21390,897,Holiday


In [ ]:
# Correlation matrix for numerical features
corr_cols = ['Unit_Price','Units_Sold','Total_Sales','Discount_Percentage',
             'Revenue','Stock_On_Hand','Store_Rating','Holiday_Flag']
corr = df[corr_cols].corr()
fig = px.imshow(corr, text_auto='.2f', aspect='auto', title='Correlation Matrix')
fig.show()

## 6. Time-Series Forecasting
The model uses **Holt-Winters Exponential Smoothing** with weekly seasonality. The final 60 days are used as test data to evaluate forecasting accuracy.

In [ ]:
# Aggregate transaction-level data to daily revenue
daily_revenue = (df.groupby('Date')['Revenue'].sum()
                   .asfreq('D')
                   .interpolate())

# Train-test split: last 60 days for evaluation
test_days = 60
train = daily_revenue.iloc[:-test_days]
test = daily_revenue.iloc[-test_days:]

model = ExponentialSmoothing(
    train,
    trend='add',
    seasonal='add',
    seasonal_periods=7,
    initialization_method='estimated'
).fit(optimized=True)

test_forecast = model.forecast(test_days)

mae = mean_absolute_error(test, test_forecast)
rmse = np.sqrt(mean_squared_error(test, test_forecast))
mape = np.mean(np.abs((test - test_forecast) / test)) * 100

print(f'MAE  : ₹{mae:,.2f}')
print(f'RMSE : ₹{rmse:,.2f}')
print(f'MAPE : {mape:.2f}%')

fig = go.Figure()
fig.add_trace(go.Scatter(x=train.index[-120:], y=train.iloc[-120:], name='Training Data'))
fig.add_trace(go.Scatter(x=test.index, y=test, name='Actual Test Revenue'))
fig.add_trace(go.Scatter(x=test_forecast.index, y=test_forecast, name='Predicted Revenue'))
fig.update_layout(title='Forecast Model Evaluation', xaxis_title='Date', yaxis_title='Daily Revenue (₹)')
fig.show()

MAE  : ₹5,582,397.32
RMSE : ₹6,744,573.91
MAPE : 10.01%


In [ ]:
# Retrain on full data and forecast next 30 days
final_model = ExponentialSmoothing(
    daily_revenue,
    trend='add',
    seasonal='add',
    seasonal_periods=7,
    initialization_method='estimated'
).fit(optimized=True)

forecast_days = 30
future_forecast = final_model.forecast(forecast_days)
forecast_df = pd.DataFrame({
    'Date': future_forecast.index,
    'Forecasted_Revenue': np.maximum(future_forecast.values, 0)
})

display(forecast_df.head(10))

fig = go.Figure()
fig.add_trace(go.Scatter(x=daily_revenue.index[-90:], y=daily_revenue.iloc[-90:], name='Historical Revenue'))
fig.add_trace(go.Scatter(x=forecast_df['Date'], y=forecast_df['Forecasted_Revenue'], name='30-Day Forecast'))
fig.update_layout(title='Future 30-Day Revenue Forecast', xaxis_title='Date', yaxis_title='Revenue (₹)')
fig.show()

,Date,Forecasted_Revenue
0,2025-01-01,"56,263,484.59"
1,2025-01-02,"56,531,959.98"
2,2025-01-03,"54,278,159.70"
3,2025-01-04,"56,893,708.98"
4,2025-01-05,"57,062,172.88"
5,2025-01-06,"55,097,436.14"
6,2025-01-07,"56,867,845.06"
7,2025-01-08,"56,254,962.82"
8,2025-01-09,"56,523,438.21"
9,2025-01-10,"54,269,637.94"


## 7. Store Segmentation using K-Means
Stores are grouped based on revenue, units sold, average rating, discount, stock, and transaction count.

In [ ]:
store_features = df.groupby(['Store_ID','Store_Location']).agg(
    Total_Revenue=('Revenue','sum'),
    Total_Units=('Units_Sold','sum'),
    Avg_Rating=('Store_Rating','mean'),
    Avg_Discount=('Discount_Percentage','mean'),
    Avg_Stock=('Stock_On_Hand','mean'),
    Transactions=('Revenue','size')
).reset_index()

feature_cols = ['Total_Revenue','Total_Units','Avg_Rating','Avg_Discount','Avg_Stock','Transactions']
scaler = StandardScaler()
scaled_features = scaler.fit_transform(store_features[feature_cols])

# Three segments: high, medium and developing performance
kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
store_features['Cluster'] = kmeans.fit_predict(scaled_features)

# Label clusters by average revenue rank
cluster_rank = store_features.groupby('Cluster')['Total_Revenue'].mean().sort_values().index.tolist()
label_map = {
    cluster_rank[0]: 'Developing Stores',
    cluster_rank[1]: 'Stable Stores',
    cluster_rank[2]: 'High-Performance Stores'
}
store_features['Segment'] = store_features['Cluster'].map(label_map)

print('Silhouette Score:', round(silhouette_score(scaled_features, store_features['Cluster']), 3))
display(store_features.sort_values('Total_Revenue', ascending=False))

fig = px.scatter(store_features, x='Total_Units', y='Total_Revenue', size='Transactions',
                 color='Segment', hover_name='Store_Location',
                 title='Store Performance Segmentation')
fig.show()

Silhouette Score: 0.36


,Store_ID,Store_Location,Total_Revenue,Total_Units,Avg_Rating,Avg_Discount,Avg_Stock,Transactions,Cluster,Segment
2,STR_103,Delhi,"5,223,064,263.84",229475,4.25,9.97,275.59,9173,0,High-Performance Stores
1,STR_102,Bangalore,"5,186,276,605.87",228384,4.25,10.00,277.11,9174,0,High-Performance Stores
0,STR_101,Mumbai,"5,169,675,266.91",226670,4.25,10.09,274.84,9039,2,Stable Stores
3,STR_104,Chennai,"5,165,937,477.75",228580,4.25,10.11,273.57,9186,2,Stable Stores
4,STR_105,Pune,"5,161,898,681.92",227715,4.25,10.08,273.83,9254,2,Stable Stores
6,STR_107,Kolkata,"5,107,832,301.73",229403,4.25,9.95,275.84,9136,0,High-Performance Stores
7,STR_108,Ahmedabad,"5,078,321,969.54",223994,4.25,9.94,272.99,9024,1,Developing Stores
5,STR_106,Hyderabad,"5,040,776,297.48",223084,4.25,9.94,273.20,9014,1,Developing Stores


## 8. Automatic AI-Style Business Insights

In [ ]:
# Identify key findings automatically from the dataset
top_category = category_summary.iloc[0]
top_store = store_summary.iloc[0]
top_region = df.groupby('Region')['Revenue'].sum().idxmax()
top_payment = df.groupby('Payment_Mode')['Revenue'].sum().idxmax()

promo_avg = df.groupby('Promotion_Applied')['Revenue'].mean()
promo_effect = ((promo_avg.get('Yes', 0) - promo_avg.get('No', 0)) / promo_avg.get('No', 1)) * 100

latest_30 = daily_revenue.iloc[-30:].sum()
previous_30 = daily_revenue.iloc[-60:-30].sum()
recent_growth = ((latest_30 - previous_30) / previous_30) * 100

forecast_total = forecast_df['Forecasted_Revenue'].sum()

insights = [
    f"1. {top_category['Product_Category']} is the highest-revenue category, generating ₹{top_category['Revenue']:,.0f}.",
    f"2. {top_store['Store_Location']} ({top_store['Store_ID']}) is the top-performing store with revenue of ₹{top_store['Revenue']:,.0f}.",
    f"3. {top_region} is the strongest region by total revenue.",
    f"4. {top_payment} contributes the highest revenue among payment modes.",
    f"5. Promotional transactions show an average revenue difference of {promo_effect:.2f}% compared with non-promotional transactions.",
    f"6. Revenue in the latest 30 days changed by {recent_growth:.2f}% compared with the previous 30 days.",
    f"7. The next 30-day forecast estimates total revenue of approximately ₹{forecast_total:,.0f}.",
    "8. Management should prioritize inventory and marketing for top categories while creating improvement plans for developing stores."
]

print('AUTOMATIC MANAGEMENT INSIGHTS')
print('-' * 45)
for insight in insights:
    print(insight)

AUTOMATIC MANAGEMENT INSIGHTS
---------------------------------------------
1. Electronics is the highest-revenue category, generating ₹8,318,538,869.
2. Delhi (STR_103) is the top-performing store with revenue of ₹5,223,064,264.
3. West is the strongest region by total revenue.
4. Credit Card contributes the highest revenue among payment modes.
5. Promotional transactions show an average revenue difference of -0.29% compared with non-promotional transactions.
6. Revenue in the latest 30 days changed by 1.19% compared with the previous 30 days.
7. The next 30-day forecast estimates total revenue of approximately ₹1,684,348,426.
8. Management should prioritize inventory and marketing for top categories while creating improvement plans for developing stores.


## 9. Export results


In [ ]:
os.makedirs('outputs', exist_ok=True)
monthly_revenue.to_csv('outputs/monthly_revenue.csv', index=False)
category_summary.to_csv('outputs/category_summary.csv', index=False)
store_summary.to_csv('outputs/store_summary.csv', index=False)
store_features.to_csv('outputs/store_segments.csv', index=False)
forecast_df.to_csv('outputs/revenue_forecast_30_days.csv', index=False)

with open('outputs/business_insights.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(insights))

print('All output files saved successfully in the outputs folder.')

All output files saved successfully in the outputs folder.


## 10. Conclusion
The project successfully converts raw retail data into a decision-support system. It provides descriptive analytics, product and store performance insights, a 30-day revenue forecast, store segmentation, and automated management recommendations. The results can help improve inventory planning, promotional decisions, resource allocation, and future revenue planning.